In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms

# Define the transformations (to load the data in normalized format)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalization
])

# Load the CIFAR-100 dataset
dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)

# Get an image and label from the dataset
image, label = dataset[0]  # Index 0, you can change this to view different images

# Undo the normalization for visualization
image = image * 0.5 + 0.5  # Scale the image back to [0, 1] range

# Convert the image tensor to a numpy array and rearrange dimensions for matplotlib
image_np = image.permute(1, 2, 0).numpy()  # Change from (C, H, W) to (H, W, C)

# Display the image with matplotlib
plt.imshow(image_np)
plt.title(f"Label: {label} (Class Index)")
plt.axis('off')
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from model import LeNet5
from tqdm import tqdm

train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Step 3: Initialize the Model, Loss, and Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5(num_classes=100).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Step 4: Train the Model
def train(model: LeNet5, train_loader: torch.utils.data.DataLoader, criterion, optimizer, device):
    model.train()
    for epoch in tqdm(range(20), "Training:"):
        #running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            #running_loss += loss.item()

        #print(f"Epoch [{epoch+1}/10], Loss: {running_loss/len(train_loader):.4f}")

def test(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Run training and testing
train(model, train_loader, criterion, optimizer, device)
test(model, test_loader, device)
